# 1 · Podaci i degradacije

Monokularna procena dubine: predvidjanje dubine svakog piksela iz jedne RGB slike.
Koristimo **NYU Depth V2** (zatvoreni prostor, dubina 0-10 m, Kinect anotacije,
654 test slike, Eigen split).

Robustnost testiramo na sinteticki degradiranim test skupovima: **homogena magla**,
**Gausovo zamucenje** i **smanjenje ekspozicije** (nocna voznja), svaka u tri jacine.

In [ ]:
import os, sys
from pathlib import Path
if Path.cwd().name == 'notebooks':
    os.chdir('..')            # run from the repo root so paths resolve
sys.path.insert(0, 'src')
sys.path.insert(0, 'scripts')

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

## Uzorak: RGB i mapa dubine

In [ ]:
test_dir = Path('data/nyu_depth_v2/official_splits/test')
rgb_path = sorted(test_dir.rglob('rgb_*.jpg'))[300]
depth_path = rgb_path.with_name(rgb_path.name.replace('rgb_', 'sync_depth_').replace('.jpg', '.png'))

rgb = np.array(Image.open(rgb_path).convert('RGB'))
depth = np.array(Image.open(depth_path), dtype=np.float32) / 1000.0   # mm -> m

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].imshow(rgb); ax[0].set_title('RGB ulaz'); ax[0].axis('off')
im = ax[1].imshow(depth, cmap='plasma'); ax[1].set_title('Dubina (m)'); ax[1].axis('off')
fig.colorbar(im, ax=ax[1], fraction=0.046)
plt.tight_layout(); plt.show()

Dubina je gusta (skoro 100% piksela). Pikseli sa vrednoscu 0 su nedostajuca merenja i
iskljucuju se maskom pri treniranju i evaluaciji.

## Tri degradacije x tri jacine

Iste transformacije koje `scripts/generate_degraded.py` upisuje na disk. Dubina se **ne**
menja - degradira se samo izgled slike, geometrija ostaje ista.

In [ ]:
import yaml
from generate_degraded import build_transform

names = ['fog', 'blur', 'exposure']
sevs = ['light', 'moderate', 'severe']
fig, axes = plt.subplots(len(names), len(sevs), figsize=(11, 9))
for r, n in enumerate(names):
    deg = yaml.safe_load(Path(f'configs/degradations/{n}.yaml').read_text())['degradation']
    for c, sv in enumerate(sevs):
        t = build_transform(deg['name'], deg['severities'][sv])
        np.random.seed(0)
        out = t(image=rgb)['image']
        axes[r, c].imshow(out)
        axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
        if r == 0:
            axes[r, c].set_title(sv)
    axes[r, 0].set_ylabel(n, fontsize=12)
plt.tight_layout(); plt.show()

**Zakljucak.** Magla smanjuje kontrast i 'gubi' udaljene objekte; zamucenje uklanja
visoke frekvencije (ivice); ekspozicija tamni scenu i gnjeci senke. Ove degradacije
simuliraju bezbednosno kriticne uslove u kojima model mora da radi.